# Stock Market Analyzer
Comprehensive Fundamental, Technical & Risk Analysis

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')


# User Input — Enter Your Stock Details

print("=" * 60)
print("STOCK MARKET ANALYZER")
print("=" * 60)

STOCK_TICKER = input("\n🔹 Enter Stock Ticker (e.g., INFY.NS, TCS.NS, AAPL): ").strip().upper()
STOCK_NAME   = input("🔹 Enter Stock Name (e.g., SBI, Apple): ").strip()

print("\n📌 Select Market:")
print("   1 → Indian Market (NSE/BSE — NIFTY 50 benchmark)")
print("   2 → US Market (NYSE/NASDAQ — S&P 500 benchmark)")
print("   3 → Other (Enter custom benchmark)")
market_choice = input("   Your choice [1/2/3] (default: 1): ").strip() or "1"

if market_choice == "2":
    BENCHMARK_TICKER = "^GSPC"
    BENCHMARK_NAME = "S&P 500"
    RISK_FREE_RATE = 0.05
    CURRENCY = "$"
elif market_choice == "3":
    BENCHMARK_TICKER = input("   Enter benchmark ticker (e.g., ^FTSE): ").strip()
    BENCHMARK_NAME = input("   Enter benchmark name (e.g., FTSE 100): ").strip()
    RISK_FREE_RATE = float(input("   Enter risk-free rate (e.g., 0.05 for 5%): ").strip() or "0.05")
    CURRENCY = input("   Enter currency symbol (e.g., $, £, €): ").strip() or "$"
else:
    BENCHMARK_TICKER = "^NSEI"
    BENCHMARK_NAME = "NIFTY 50"
    RISK_FREE_RATE = 0.07
    CURRENCY = "₹"

print("\n📅 Select Analysis Period:")
print("   1 → 1 Year    2 → 2 Years    3 → 3 Years")
print("   5 → 5 Years   10 → 10 Years  max → Maximum")
period_input = input("   Your choice (default: 5): ").strip() or "5"
PERIOD_MAP = {"1": "1y", "2": "2y", "3": "3y", "5": "5y", "10": "10y", "max": "max"}
PERIOD = PERIOD_MAP.get(period_input, period_input + "y" if period_input.isdigit() else "5y")

print(f"\n{'─'*60}")
print(f"  🎯 Stock     : {STOCK_NAME} ({STOCK_TICKER})")
print(f"  📊 Benchmark : {BENCHMARK_NAME} ({BENCHMARK_TICKER})")
print(f"  📅 Period    : {PERIOD}")
print(f"  💰 Currency  : {CURRENCY}")
print(f"  📉 Risk-Free : {RISK_FREE_RATE*100:.1f}%")
print(f"{'─'*60}")
confirm = input("\n✅ Proceed with analysis? [Y/n]: ").strip().lower()
if confirm == 'n':
    print("❌ Analysis cancelled.")
    raise SystemExit


# Fetch Data

print(f"📥 Fetching data for {STOCK_NAME} ({STOCK_TICKER})...")
stock = yf.Ticker(STOCK_TICKER)
benchmark = yf.Ticker(BENCHMARK_TICKER)

hist = stock.history(period=PERIOD)
bench_hist = benchmark.history(period=PERIOD)
info = stock.info

if hist.empty:
    raise ValueError(f"❌ No data found for {STOCK_TICKER}. Check the ticker symbol.")

print(f"✅ Loaded {len(hist)} trading days of data")
print(f"📅 Date range: {hist.index[0].strftime('%Y-%m-%d')} to {hist.index[-1].strftime('%Y-%m-%d')}")


# Business Understanding

def print_business_overview(info):
    print("=" * 70)
    print("🔹 1. BUSINESS UNDERSTANDING")
    print("=" * 70)
    print(f"  Company    : {info.get('longName', STOCK_NAME)}")
    print(f"  Sector     : {info.get('sector', 'N/A')}")
    print(f"  Industry   : {info.get('industry', 'N/A')}")
    print(f"  Country    : {info.get('country', 'N/A')}")
    print(f"  Employees  : {info.get('fullTimeEmployees', 'N/A'):,}" if isinstance(info.get('fullTimeEmployees'), int) else f"  Employees  : N/A")
    print(f"  Market Cap : {CURRENCY}{info.get('marketCap', 0)/1e9:.2f}B" if info.get('marketCap') else "  Market Cap : N/A")
    summary = info.get('longBusinessSummary', 'No summary available.')
    print(f"\n  📝 Summary:\n  {summary[:500]}...")
    print()

print_business_overview(info)


# Fundamental Analysis

def fundamental_analysis(stock, info):
    print("=" * 70)
    print("🔹 2. FUNDAMENTAL ANALYSIS")
    print("=" * 70)

    fundamentals = {}


    pe = info.get('trailingPE', info.get('forwardPE'))
    pb = info.get('priceToBook')
    de = info.get('debtToEquity')
    roe = info.get('returnOnEquity')
    margin = info.get('profitMargins')
    eps = info.get('trailingEps')
    rev_growth = info.get('revenueGrowth')
    earn_growth = info.get('earningsGrowth')
    div_yield = info.get('dividendYield')

    print(f"  P/E Ratio        : {pe:.2f}" if pe else "  P/E Ratio        : N/A")
    print(f"  P/B Ratio        : {pb:.2f}" if pb else "  P/B Ratio        : N/A")
    print(f"  Debt/Equity      : {de:.2f}" if de else "  Debt/Equity      : N/A")
    print(f"  ROE              : {roe*100:.2f}%" if roe else "  ROE              : N/A")
    print(f"  Profit Margin    : {margin*100:.2f}%" if margin else "  Profit Margin    : N/A")
    print(f"  EPS              : {eps:.2f}" if eps else "  EPS              : N/A")
    print(f"  Revenue Growth   : {rev_growth*100:.2f}%" if rev_growth else "  Revenue Growth   : N/A")
    print(f"  Earnings Growth  : {earn_growth*100:.2f}%" if earn_growth else "  Earnings Growth  : N/A")
    print(f"  Dividend Yield   : {div_yield*100:.2f}%" if div_yield else "  Dividend Yield   : N/A")


    try:
        financials = stock.financials
        if financials is not None and not financials.empty:
            revenue = financials.loc['Total Revenue'] if 'Total Revenue' in financials.index else None
            net_income = financials.loc['Net Income'] if 'Net Income' in financials.index else None

            if revenue is not None:
                print(f"\n  📊 Revenue (Last {len(revenue)} years):")
                for date, val in revenue.items():
                    print(f"     {date.strftime('%Y')}: {CURRENCY}{val/1e9:.2f}B")
                fundamentals['revenue'] = revenue

            if net_income is not None:
                print(f"\n  📊 Net Profit (Last {len(net_income)} years):")
                for date, val in net_income.items():
                    print(f"     {date.strftime('%Y')}: {CURRENCY}{val/1e9:.2f}B")
                fundamentals['net_income'] = net_income
    except Exception as e:
        print(f"  ⚠️ Could not fetch financials: {e}")


    score = 10  # base
    if roe and roe > 0.15: score += 3
    elif roe and roe > 0.10: score += 1
    elif roe and roe < 0.05: score -= 3

    if de is not None and de < 50: score += 2
    elif de is not None and de > 100: score -= 2

    if margin and margin > 0.15: score += 2
    elif margin and margin < 0.05: score -= 2

    if rev_growth and rev_growth > 0.10: score += 2
    elif rev_growth and rev_growth < 0: score -= 2

    if pe and pe < 25: score += 1
    elif pe and pe > 50: score -= 1

    score = max(0, min(20, score))
    strength = "Strong 💪" if score >= 14 else "Average ⚖️" if score >= 8 else "Weak ⚠️"
    print(f"\n  🏆 Fundamental Score: {score}/20 ({strength})")

    fundamentals.update({'score': score, 'strength': strength, 'pe': pe, 'roe': roe, 'de': de, 'margin': margin})
    return fundamentals

fund_results = fundamental_analysis(stock, info)


# Technical Analysis

def technical_analysis(hist):
    print("\n" + "=" * 70)
    print("🔹 3. TECHNICAL ANALYSIS")
    print("=" * 70)

    df = hist.copy()
    df['SMA_50'] = df['Close'].rolling(window=50).mean()
    df['SMA_200'] = df['Close'].rolling(window=200).mean()
    df['EMA_20'] = df['Close'].ewm(span=20).mean()
    df['Daily_Return'] = df['Close'].pct_change()


    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))


    ema12 = df['Close'].ewm(span=12).mean()
    ema26 = df['Close'].ewm(span=26).mean()
    df['MACD'] = ema12 - ema26
    df['MACD_Signal'] = df['MACD'].ewm(span=9).mean()


    df['BB_Mid'] = df['Close'].rolling(20).mean()
    bb_std = df['Close'].rolling(20).std()
    df['BB_Upper'] = df['BB_Mid'] + 2 * bb_std
    df['BB_Lower'] = df['BB_Mid'] - 2 * bb_std

    current = df['Close'].iloc[-1]
    sma50 = df['SMA_50'].iloc[-1]
    sma200 = df['SMA_200'].iloc[-1]
    rsi = df['RSI'].iloc[-1]


    if current > sma50 > sma200:
        trend = "🟢 BULLISH"
    elif current < sma50 < sma200:
        trend = "🔴 BEARISH"
    else:
        trend = "🟡 SIDEWAYS"

    golden_cross = sma50 > sma200
    print(f"  Current Price : {CURRENCY}{current:.2f}")
    print(f"  50-day SMA    : {CURRENCY}{sma50:.2f}")
    print(f"  200-day SMA   : {CURRENCY}{sma200:.2f}")
    print(f"  RSI (14)      : {rsi:.2f}")
    print(f"  Trend         : {trend}")
    print(f"  Golden Cross  : {'Yes ✅' if golden_cross else 'No ❌'}")


    recent = df['Close'].tail(60)
    support = recent.min()
    resistance = recent.max()
    print(f"  Support  (60d): {CURRENCY}{support:.2f}")
    print(f"  Resistance(60d): {CURRENCY}{resistance:.2f}")

    score = 10
    if "BULLISH" in trend: score += 4
    elif "BEARISH" in trend: score -= 4
    if golden_cross: score += 2
    if 40 <= rsi <= 60: score += 2
    elif rsi > 70: score -= 2
    elif rsi < 30: score += 1
    if current > sma200: score += 2
    elif current < sma200: score -= 2

    score = max(0, min(20, score))
    print(f"\n  🏆 Technical Score: {score}/20")

    return df, {'score': score, 'trend': trend, 'rsi': rsi, 'sma50': sma50,
                'sma200': sma200, 'support': support, 'resistance': resistance,
                'current': current, 'golden_cross': golden_cross}

df, tech_results = technical_analysis(hist)


# MACHINE LEARNING MODEL (Price Prediction)


from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

print("\n" + "=" * 70)
print("🔹  ML MODEL: STOCK PRICE PREDICTION")
print("=" * 70)


ml_df = df.copy()

ml_df['Return'] = ml_df['Close'].pct_change()
ml_df['Volatility'] = ml_df['Return'].rolling(10).std()
ml_df['Target'] = ml_df['Close'].shift(-1)  # Next day price

ml_df = ml_df.dropna()

features = ['Close', 'SMA_50', 'SMA_200', 'EMA_20', 'RSI', 'Return', 'Volatility']
X = ml_df[features]
y = ml_df['Target']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)


model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"📉 MAE (Error): {mae:.2f}")
print(f"📊 R² Score  : {r2:.2f}")


latest_data = X.iloc[-1:].values
next_price = model.predict(latest_data)[0]

print(f"\n🔮 Predicted Next Day Price: {CURRENCY}{next_price:.2f}")
print(f"📍 Current Price           : {CURRENCY}{df['Close'].iloc[-1]:.2f}")


importances = model.feature_importances_

print("\n📊 Feature Importance:")
for f, imp in zip(features, importances):
    print(f"  {f}: {imp:.3f}")

# Volatility & Risk Analysis

def risk_analysis(df, bench_hist):
    print("\n" + "=" * 70)
    print("🔹 4. VOLATILITY & RISK ANALYSIS")
    print("=" * 70)

    returns = df['Close'].pct_change().dropna()
    vol_daily = returns.std()
    vol_annual = vol_daily * np.sqrt(252)

    bench_returns = bench_hist['Close'].pct_change().dropna()
    common_idx = returns.index.intersection(bench_returns.index)

    if len(common_idx) > 30:
        stock_r = returns.loc[common_idx]
        bench_r = bench_returns.loc[common_idx]
        cov = np.cov(stock_r, bench_r)
        beta = cov[0, 1] / cov[1, 1]
    else:
        beta = None

    max_dd = ((df['Close'] / df['Close'].cummax()) - 1).min()
    sharpe = (returns.mean() * 252 - RISK_FREE_RATE) / vol_annual if vol_annual > 0 else 0
    var_95 = np.percentile(returns, 5)

    risk_class = "Low 🟢" if vol_annual < 0.20 else "Medium 🟡" if vol_annual < 0.35 else "High 🔴"

    print(f"  Daily Volatility   : {vol_daily*100:.2f}%")
    print(f"  Annual Volatility  : {vol_annual*100:.2f}%")
    print(f"  Beta vs {BENCHMARK_NAME}  : {beta:.2f}" if beta else f"  Beta               : N/A")
    print(f"  Sharpe Ratio       : {sharpe:.2f}")
    print(f"  Max Drawdown       : {max_dd*100:.2f}%")
    print(f"  VaR (95%)          : {var_95*100:.2f}%")
    print(f"  Risk Level         : {risk_class}")

    score = 10
    if vol_annual < 0.20: score += 4
    elif vol_annual < 0.30: score += 2
    elif vol_annual > 0.40: score -= 4
    if beta and beta < 1: score += 2
    elif beta and beta > 1.5: score -= 2
    if sharpe > 1: score += 2
    elif sharpe < 0: score -= 2
    if max_dd > -0.20: score += 2
    elif max_dd < -0.40: score -= 2

    score = max(0, min(20, score))
    print(f"\n  🏆 Risk Score: {score}/20")

    return {'score': score, 'vol_annual': vol_annual, 'beta': beta, 'sharpe': sharpe,
            'max_dd': max_dd, 'risk_class': risk_class, 'returns': returns,
            'bench_returns': bench_returns, 'var_95': var_95}

risk_results = risk_analysis(df, bench_hist)

# Market Comparison

def market_comparison(hist, bench_hist):
    print("\n" + "=" * 70)
    print("🔹 5. MARKET COMPARISON")
    print("=" * 70)

    stock_ret_1y = (hist['Close'].iloc[-1] / hist['Close'].iloc[-252] - 1) * 100 if len(hist) >= 252 else None
    bench_ret_1y = (bench_hist['Close'].iloc[-1] / bench_hist['Close'].iloc[-252] - 1) * 100 if len(bench_hist) >= 252 else None

    stock_ret_ytd = (hist['Close'].iloc[-1] / hist['Close'][hist.index >= f"{datetime.now().year}-01-01"].iloc[0] - 1) * 100 if len(hist[hist.index >= f"{datetime.now().year}-01-01"]) > 0 else None

    if stock_ret_1y and bench_ret_1y:
        alpha = stock_ret_1y - bench_ret_1y
        perf = "Outperforming 🚀" if alpha > 0 else "Underperforming 📉"
        print(f"  Stock 1Y Return : {stock_ret_1y:.2f}%")
        print(f"  {BENCHMARK_NAME} 1Y Return : {bench_ret_1y:.2f}%")
        print(f"  Alpha (1Y)      : {alpha:+.2f}%")
        print(f"  Performance     : {perf}")
    else:
        alpha = 0
        perf = "N/A"

    if stock_ret_ytd:
        print(f"  Stock YTD Return: {stock_ret_ytd:.2f}%")

    score = 10
    if alpha > 10: score += 5
    elif alpha > 5: score += 3
    elif alpha > 0: score += 1
    elif alpha < -10: score -= 4
    elif alpha < -5: score -= 2

    if stock_ret_1y and stock_ret_1y > 15: score += 3
    elif stock_ret_1y and stock_ret_1y < 0: score -= 3

    score = max(0, min(20, score))
    print(f"\n  🏆 Market Strength Score: {score}/20")

    return {'score': score, 'stock_ret_1y': stock_ret_1y, 'bench_ret_1y': bench_ret_1y,
            'alpha': alpha, 'perf': perf}

mkt_results = market_comparison(hist, bench_hist)


# Growth Potential Score

def growth_score(info, fund_results, tech_results):
    score = 10
    rev_g = info.get('revenueGrowth')
    earn_g = info.get('earningsGrowth')
    if rev_g and rev_g > 0.15: score += 3
    elif rev_g and rev_g > 0.05: score += 1
    elif rev_g and rev_g < 0: score -= 3
    if earn_g and earn_g > 0.15: score += 3
    elif earn_g and earn_g > 0.05: score += 1
    elif earn_g and earn_g < 0: score -= 3
    roe = info.get('returnOnEquity')
    if roe and roe > 0.20: score += 2
    if "BULLISH" in tech_results['trend']: score += 2
    elif "BEARISH" in tech_results['trend']: score -= 2
    return max(0, min(20, score))

growth_sc = growth_score(info, fund_results, tech_results)
print(f"\n  🏆 Growth Potential Score: {growth_sc}/20")


# Final Scoring & Verdict

def final_verdict(fund, tech, risk, market, growth):
    total = fund['score'] + tech['score'] + risk['score'] + market['score'] + growth

    print("\n" + "=" * 70)
    print("🔹 6. SCORING SYSTEM")
    print("=" * 70)
    print(f"  Fundamentals   : {fund['score']}/20")
    print(f"  Technical Trend: {tech['score']}/20")
    print(f"  Risk Level     : {risk['score']}/20")
    print(f"  Growth Potential: {growth}/20")
    print(f"  Market Strength: {market['score']}/20")
    print(f"  {'─'*30}")
    print(f"  TOTAL SCORE    : {total}/100")

    print("\n" + "=" * 70)
    print("🔹 7. FINAL DECISION")
    print("=" * 70)

    trend = tech['trend']
    vol = "Low" if risk['vol_annual'] < 0.20 else "Medium" if risk['vol_annual'] < 0.35 else "High"

    # Long-term
    if total >= 70:
        lt = "✅ BUY — Strong fundamentals and growth outlook"
    elif total >= 50:
        lt = "⏸️ HOLD — Decent but monitor for improvement"
    else:
        lt = "❌ AVOID — Weak metrics, high risk"

    # Short-term
    if "BULLISH" in trend and tech['rsi'] < 70:
        st = "✅ BUY — Bullish momentum, not overbought"
    elif "BEARISH" in trend:
        st = "🔻 SELL / WAIT — Bearish trend active"
    elif tech['rsi'] > 70:
        st = "⚠️ WAIT — Overbought, wait for pullback"
    else:
        st = "⏸️ WAIT — No clear signal"

    risk_lbl = risk['risk_class']

    print(f"\n  📊 Stock       : {STOCK_NAME} ({STOCK_TICKER})")
    print(f"  📈 Trend       : {trend}")
    print(f"  📉 Volatility  : {vol}")
    print(f"  💰 Fundamentals: {fund['strength']}")
    print(f"  🎯 Score       : {total}/100")
    print(f"\n  🟢 Long Term   : {lt}")
    print(f"  🔵 Short Term  : {st}")
    print(f"  ⚡ Risk Level  : {risk_lbl}")

    return total

total_score = final_verdict(fund_results, tech_results, risk_results, mkt_results, growth_sc)


# Visualizations

plt.style.use('dark_background')
sns.set_palette("husl")
fig = plt.figure(figsize=(22, 28))
fig.suptitle(f'📊 {STOCK_NAME} ({STOCK_TICKER}) — Comprehensive Analysis',
             fontsize=22, fontweight='bold', color='#00BFFF', y=0.98)


ax1 = fig.add_subplot(4, 2, 1)
ax1.plot(df.index, df['Close'], label='Close', color='#00BFFF', linewidth=1.5)
ax1.plot(df.index, df['SMA_50'], label='SMA 50', color='#FFD700', linewidth=1, linestyle='--')
ax1.plot(df.index, df['SMA_200'], label='SMA 200', color='#FF6347', linewidth=1, linestyle='--')
ax1.fill_between(df.index, df['BB_Lower'], df['BB_Upper'], alpha=0.1, color='cyan')
ax1.set_title('Price & Moving Averages', fontsize=12, fontweight='bold')
ax1.legend(fontsize=8)
ax1.grid(alpha=0.2)


ax2 = fig.add_subplot(4, 2, 2)
colors = ['#00FF7F' if df['Close'].iloc[i] >= df['Close'].iloc[i-1] else '#FF4500'
          for i in range(1, len(df))]
colors.insert(0, '#00FF7F')
ax2.bar(df.index, df['Volume'], color=colors, alpha=0.7, width=1.5)
ax2.set_title('Volume', fontsize=12, fontweight='bold')
ax2.grid(alpha=0.2)


ax3 = fig.add_subplot(4, 2, 3)
ax3.plot(df.index, df['RSI'], color='#DA70D6', linewidth=1.2)
ax3.axhline(70, color='red', linestyle='--', alpha=0.7, label='Overbought (70)')
ax3.axhline(30, color='green', linestyle='--', alpha=0.7, label='Oversold (30)')
ax3.fill_between(df.index, 30, 70, alpha=0.05, color='yellow')
ax3.set_title('RSI (14)', fontsize=12, fontweight='bold')
ax3.set_ylim(0, 100)
ax3.legend(fontsize=8)
ax3.grid(alpha=0.2)


ax4 = fig.add_subplot(4, 2, 4)
ax4.plot(df.index, df['MACD'], label='MACD', color='#00BFFF', linewidth=1)
ax4.plot(df.index, df['MACD_Signal'], label='Signal', color='#FF6347', linewidth=1)
macd_hist = df['MACD'] - df['MACD_Signal']
ax4.bar(df.index, macd_hist, color=['#00FF7F' if v >= 0 else '#FF4500' for v in macd_hist], alpha=0.4, width=1.5)
ax4.set_title('MACD', fontsize=12, fontweight='bold')
ax4.legend(fontsize=8)
ax4.grid(alpha=0.2)


ax5 = fig.add_subplot(4, 2, 5)
returns = df['Close'].pct_change().dropna()
ax5.hist(returns, bins=80, color='#00BFFF', alpha=0.7, edgecolor='none', density=True)
ax5.axvline(returns.mean(), color='#FFD700', linestyle='--', label=f'Mean: {returns.mean()*100:.2f}%')
ax5.axvline(np.percentile(returns, 5), color='red', linestyle='--', label=f'VaR 5%: {np.percentile(returns,5)*100:.2f}%')
ax5.set_title('Returns Distribution', fontsize=12, fontweight='bold')
ax5.legend(fontsize=8)
ax5.grid(alpha=0.2)


ax6 = fig.add_subplot(4, 2, 6)
common_start = max(hist.index[0], bench_hist.index[0])
s = hist['Close'][hist.index >= common_start]
b = bench_hist['Close'][bench_hist.index >= common_start]
ax6.plot(s.index, s / s.iloc[0] * 100, label=STOCK_NAME, color='#00BFFF', linewidth=1.5)
ax6.plot(b.index, b / b.iloc[0] * 100, label=BENCHMARK_NAME, color='#FFD700', linewidth=1.5)
ax6.set_title(f'{STOCK_NAME} vs {BENCHMARK_NAME} (Normalized)', fontsize=12, fontweight='bold')
ax6.legend(fontsize=8)
ax6.grid(alpha=0.2)

ax7 = fig.add_subplot(4, 2, 7)
dd = (df['Close'] / df['Close'].cummax()) - 1
ax7.fill_between(dd.index, dd, 0, color='#FF4500', alpha=0.5)
ax7.set_title('Drawdown', fontsize=12, fontweight='bold')
ax7.set_ylabel('Drawdown %')
ax7.grid(alpha=0.2)


ax8 = fig.add_subplot(4, 2, 8)
categories = ['Fundamentals', 'Technical', 'Risk', 'Growth', 'Market']
scores = [fund_results['score'], tech_results['score'], risk_results['score'], growth_sc, mkt_results['score']]
colors_bar = ['#00BFFF', '#FFD700', '#FF6347', '#00FF7F', '#DA70D6']
bars = ax8.barh(categories, scores, color=colors_bar, edgecolor='white', linewidth=0.5, height=0.6)
ax8.set_xlim(0, 20)
ax8.set_title(f'Score Breakdown (Total: {total_score}/100)', fontsize=12, fontweight='bold')
for bar, sc in zip(bars, scores):
    ax8.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, f'{sc}/20',
             va='center', fontsize=10, fontweight='bold', color='white')
ax8.grid(alpha=0.2, axis='x')

plt.tight_layout(rect=[0, 0, 1, 0.96])
filename = f'stock_analysis_{STOCK_TICKER.replace(".", "_").replace("^", "")}.png'
plt.savefig(filename, dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

print("\n" + "=" * 60)
print(f"✅ Analysis complete for {STOCK_NAME} ({STOCK_TICKER})!")
print(f"📊 Chart saved as '{filename}'")
print(f"🎯 Final Score: {total_score}/100")
print("=" * 60)
print("\n💡 To analyze another stock, simply re-run this notebook!")


Result
STOCK MARKET ANALYZER
